# MedXIR document rerank on the hybrid pool -> nb60 submissions with new document lists

Generated by `scripts/build_doc_rerank_notebook.py`. Edit the generator, not this file.

RTX Pro 6000, Internet OFF. Inputs (found by glob under `/kaggle/input`): `medxir-chunk-corpus`,
`medxir-package-rerankers`, and the hybrid run's `pool.parquet` + `medxir-hybrid-submission` (zip or json).

Steps: load pool and chunks (expand_chunks.load_inputs) -> candidate documents per query (submitted 200, then
pool order) -> pairs (query, lead + best chunk) -> each reranker in DOC_RERANKERS -> `doc_rank.parquet` ->
nb60 ZIPs with RRF-fused document lists + `expand_manifest.json`.

Time: bge-reranker-v2-m3 on ranks 0-400 (~460k pairs at 512 tokens) took 25 min on 29/10 (FINAL 0.2253).
Qwen3-Reranker-4B runs ~40 pairs/s, so it scores only the zone around the cut (ranks 120-360, ~2 h); attach
the bge run's doc_rank.parquet to fuse both. Progress and ETA are in progress.txt.

In [ ]:
# ===================== Config: the only cell to edit =====================
# reranker -> (first, last) candidate rank scored per query (submitted order; the cut is at 200).
# bge on 0-400: 29/10 run 0.2253 (nb60 0.2235). Qwen3-Reranker-4B on 120-360 (~2 h) lost: qwen4b-only 0.2176.
DOC_RERANKERS = {}          # CPU rebuild below; GPU run: {'bge-reranker-v2-m3': (0, 400)}
# DOC_RERANKERS = {'Qwen3-Reranker-4B': (120, 360)}   # 30/10 run; the 'qwen4b' variants keep the top 120
# CPU rebuild (no GPU, ~15 min): DOC_RERANKERS = {}, PRIOR_DOC_RANK = True, attach the earlier run's doc_rank.parquet
PRIOR_DOC_RANK = True       # True: merge the score columns of an attached doc_rank.parquet of the SAME hybrid run
LEAD_CHARS = 400            # characters of the document start (title / lead) shown before the chunks
DOC_CHUNKS = 1              # best pool chunks of the document shown; 2 doubles the pair length (Qwen ~4 h)
DOC_MAX_TOKENS = 1024 if DOC_CHUNKS > 1 else 512   # pair length cap for the reranker
QWEN_BATCH = 16
BUILD = ['nb60-drr-bge-only-ghigh', 'nb60-drr-bge-only-ghigh-cover', 'nb30-drr-bge-only-ghigh-cover']
# v2 30/10: bge-only 0.2464, glow 0.2352, ghigh 0.2477
QUERY_LIMIT = 0             # >0: first N queries only (speed check; doc_rank.parquet written, no ZIPs)

In [ ]:
import faulthandler
import json
import os
import resource
import time
import traceback
from contextlib import contextmanager
from pathlib import Path

OUT = Path('/kaggle/working')
OUT.mkdir(parents=True, exist_ok=True)
T0 = time.perf_counter()
PROGRESS = OUT / 'progress.txt'
FAULT_FILE = open(OUT / 'faulthandler.txt', 'w')  # native crashes dump the Python stack here
faulthandler.enable(file=FAULT_FILE, all_threads=True)
FAILED = None


def note(message):
    rss_gib = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2 ** 20  # Linux reports KiB
    line = f'{time.perf_counter() - T0:8.1f}s  peak_rss={rss_gib:.1f}GiB  {message}'
    print(line, flush=True)
    with open(PROGRESS, 'a', encoding='utf-8') as file:
        file.write(line + '\n')


@contextmanager
def stage(name):
    # A Python exception is written to retrieval_error.txt and does not stop the kernel, so the
    # session ends COMPLETE and the files can be fetched; later steps are skipped.
    global FAILED
    note(f'start {name}')
    try:
        yield
        note(f'done {name}')
    except Exception:
        FAILED = name
        (OUT / 'retrieval_error.txt').write_text(f'stage: {name}\n\n{traceback.format_exc()}', encoding='utf-8')
        note(f'FAILED {name}; traceback in retrieval_error.txt')

In [ ]:
# Copied from scripts/expand_chunks.py by scripts/build_doc_rerank_notebook.py. Edit the script and rebuild.
"""Rebuild the chunk lists of the hybrid submission with neighbouring chunks merged into longer spans.

A reference passage counts as found when a submitted text of the same document covers >= 40% of it (token
LCS). Chunk recall (0.18) is far below document recall (0.47): the right documents are found, the passages
are not. Here the top-ranked chunks bring their neighbours (order - 1, order + 1) along, and contiguous
chunks of one text are merged into one span (the 50-word overlaps are dropped once). The byte budget per
query stays that of the hybrid submission, so lower-ranked chunks make room for the neighbours.

The document lists are kept as submitted; documents of new spans are appended (as in assemble()).
Optional per variant: n_docs extends the document list with the next pool documents (best chunk first);
quota reserves that share of the chunk bytes for ZH/EN texts; doc_quota swaps the lowest VI documents of the
list for the best ZH/EN pool documents (both lost on the leaderboard, 29/10); doc_rr re-orders the
candidate documents by weighted RRF of their submitted rank and document-reranker scores (doc_rank.parquet,
written by the doc-rerank notebook, kaggle_doc_rerank.ipynb), so the reranker decides which make the cut.
conf_gamma moves chunk bytes between queries, total unchanged: a query's confidence is its number of pool
chunks with rerank score >= conf_t; confident queries lose budget (their lowest-ranked chunks go) and the
budget goes to queries with few strong chunks, which reach deeper into their pool.
Runs on a Kaggle CPU session (Internet OFF is fine). Inputs:
  medxir-chunk-corpus      chunks/*/part-*.parquet, hash_groups.parquet
  medxir-hybrid-retrieval  pool.parquet, medxir-hybrid-submission.zip   (notebook output or uploaded dataset)
  doc_rank.parquet         only for doc_rr variants (in OUT_DIR or an input)
Output: one ZIP per variant in /kaggle/working (or OUT_DIR), plus expand_manifest.json.
"""
import glob
import json
import math
import os
import zipfile
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

# nb: number of top chunks whose neighbours are added (0 = only merge, no new neighbours);
#     'whole' = probe: full texts of the top documents instead of ranked chunks.
# n_docs: document list length (0 = as submitted). quota / doc_quota: min ZH/EN share of chunk bytes / documents.
# doc_rr: {score column of doc_rank.parquet: RRF weight}; w_base: RRF weight of the submitted document rank;
# keep_top: that many top candidates kept as submitted, the rest of the list re-ordered.
# conf_gamma: budget weight (confidence / median confidence) ** -gamma, clipped to CONF_CLIP, renormalised.
# min_chars: spans shorter than this are not submitted. A span matches when it covers 40% of a reference
# passage, so a 5-character ZH entry ('食管炎') can only match an equally short passage; such spans were 11% of
# the entries of the 0.2464 ZIP (< 200 chars) for 0.35% of its text.
# min_score: ranked chunks with a rerank score below it are not taken (nor their neighbours); the bytes a query
# leaves unused go to the queries that used their whole budget, by the same confidence weights. bge scores are
# high: at 0.2 the cut removes ~46 chunks in 16% of the queries (the unsure ones) and none elsewhere.
# zip: 'bzip2' writes the ZIP with bzip2 instead of deflate: the 0.2477 JSON is 86.5 MB deflated, 56.5 MB with
# bzip2 (LZMA 54.4 MB, 10x slower), so ~1.5x more text fits under the 100 MB limit. budget_scale replaces
# BUDGET_SCALE; the size loop shrinks it until the ZIP fits. Under the limit chunk recall grows with the bytes
# (each gold document holds ~1 reference passage, so more bytes = more documents reached).
# cover (first, last): after the nb top chunks (with neighbours), the best pool chunk of every listed document
# at ranks first..last that has none yet, then the rest of the pool. 30/10 (Qwen-8B, 0.2509): ranks 100-200 of
# the document list got chunk text for 10% of the documents, ranks 50-100 for 48%, while ~15% of them are
# relevant and a relevant document holds ~1 reference passage.
VARIANTS = {
    'nb0': dict(nb=0), 'nb10': dict(nb=10), 'nb30': dict(nb=30), 'nb60': dict(nb=60), 'wholedoc': dict(nb='whole'),
    'nb60-d300': dict(nb=60, n_docs=300),
    'nb60-q20': dict(nb=60, quota=0.2),
    'nb60-q20-dq20': dict(nb=60, quota=0.2, doc_quota=0.2),
    'nb60-drr-bge': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=1.0),
    'nb60-drr-bge-only': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0),
    # Qwen3-Reranker-4B scores only the zone around the cut (ranks 120-360): the top 120 stay as submitted.
    'nb60-drr-qwen4b': dict(nb=60, doc_rr={'Qwen3-Reranker-4B': 1.0}, w_base=1.0, keep_top=120),
    'nb60-drr-qwen4b-only': dict(nb=60, doc_rr={'Qwen3-Reranker-4B': 1.0}, w_base=0.0, keep_top=120),
    'nb60-drr-both': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0, 'Qwen3-Reranker-4B': 1.0}, w_base=1.0,
                          keep_top=120),
    # Bge-only document list, chunk bytes moved between queries. 30/10 on v2 (bge-only 0.2464): glow (to unsure
    # queries) 0.2352, ghigh (to confident queries) 0.2477; macro F2 rewards the queries that already score.
    'nb60-drr-bge-only-glow': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0, conf_gamma=0.5, conf_t=0.5),
    'nb60-drr-bge-only-ghigh': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0, conf_gamma=-0.5, conf_t=0.5),
    'nb60-drr-bge-only-ghigh1': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0, conf_gamma=-1.0,
                                     conf_t=0.5),
    # Precision: no tiny spans; 'nball' gives every ranked chunk its neighbours (no single chunks filling the
    # budget), the entry shape of the x1-c70 submission (Chunk P 0.208 vs 0.155).
    'nb60-drr-bge-only-ghigh1-min200': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0, conf_gamma=-1.0,
                                            conf_t=0.5, min_chars=200),
    'nb60-drr-bge-only-ghigh-min200-s20': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0,
                                               conf_gamma=-0.5, conf_t=0.5, min_chars=200, min_score=0.2),
    'nb60-drr-bge-only-ghigh1-min200-s20': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0,
                                                conf_gamma=-1.0, conf_t=0.5, min_chars=200, min_score=0.2),
    'nb60-drr-bge-only-ghigh-bz': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0, conf_gamma=-0.5,
                                       conf_t=0.5, zip='bzip2', budget_scale=1.6),
    'nb60-drr-bge-only-ghigh-cover': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0, conf_gamma=-0.5,
                                          conf_t=0.5, cover=(20, 200)),
    'nb30-drr-bge-only-ghigh-cover': dict(nb=30, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0, conf_gamma=-0.5,
                                          conf_t=0.5, cover=(20, 200)),
    'nb60-drr-bge-only-ghigh-min200': dict(nb=60, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0, conf_gamma=-0.5,
                                           conf_t=0.5, min_chars=200),
    'nball-drr-bge-only-ghigh-min200': dict(nb=10 ** 6, doc_rr={'bge-reranker-v2-m3': 1.0}, w_base=0.0,
                                            conf_gamma=-0.5, conf_t=0.5, min_chars=200),
}
ONLY = ['nb60-d300']   # variants to build this run; [] = all
BUDGET_SCALE = 1.08   # 87.5 MB hybrid ZIP * 1.08 stays below the 100 MB limit
MAX_ZIP_MB = 99.0
CONF_CLIP = (0.5, 2.0)
RRF_K = 60
INPUT = os.environ.get('INPUT_ROOT', '/kaggle/input')
OUT = Path(os.environ.get('OUT_DIR', '/kaggle/working'))


def one_match(pattern):
    hits = sorted(p for p in glob.glob(f'{INPUT}/**/{pattern}', recursive=True))
    if len(hits) != 1:
        for root in sorted(glob.glob(f'{INPUT}/*/*/*')) or sorted(glob.glob(f'{INPUT}/*')):
            print('input:', root, sorted(os.listdir(root))[:8] if os.path.isdir(root) else '')
    assert len(hits) == 1, f'{pattern}: expected one match, found {hits[:10]}'
    return Path(hits[0])


def load_submission():
    # An uploaded dataset is unzipped by Kaggle: accept the JSON too.
    hits = sorted(glob.glob(f'{INPUT}/**/medxir-hybrid-submission.json', recursive=True))
    if hits:
        return json.loads(Path(hits[0]).read_text(encoding='utf-8'))
    with zipfile.ZipFile(one_match('medxir-hybrid-submission.zip')) as archive:
        return json.loads(archive.read(archive.namelist()[0]))


def merge_run(rows):
    """One text out of chunks of one document text with consecutive orders (rows sorted by order)."""
    text, end = rows[0]['text'], rows[0]['end']
    for row in rows[1:]:
        text += row['text'][max(0, end - row['start']):]
        end = row['end']
    return text


def build_query(ranked, n_neighbour, budget, rows, by_pos, docs_of, quota=0.0, scores=None, min_score=0.0):
    """Selected chunk ids for one query: ranked chunks (+ neighbours of the top ones) until the budget is used.

    quota > 0: a first pass caps VI chunks at (1 - quota) of the budget so ZH/EN chunks further down the pool
    get the rest; a second pass fills what ZH/EN could not use.
    """
    chosen, used, vi_used = [], 0, 0
    for vi_cap in ([budget * (1 - quota)] if quota else []) + [budget]:
        for rank, cid in enumerate(ranked):
            if min_score and scores[rank] < min_score:
                break   # pool is in score order
            group = [cid]
            if rank < n_neighbour:
                sha, order = rows[cid]['sha'], rows[cid]['order']
                group += [by_pos[(sha, o)] for o in (order - 1, order + 1) if (sha, o) in by_pos]
            for c in group:
                if c in chosen:
                    continue
                cost = len(rows[c]['text'].encode('utf-8')) * len(docs_of(c))
                vi = rows[c].get('lang', 'vi') == 'vi'
                if used + cost > budget or (vi and vi_used + cost > vi_cap):
                    continue  # a long or wide-fanout chunk does not fit; smaller ones further down may
                chosen.append(c)
                used += cost
                vi_used += cost if vi else 0
    return chosen


def doc_list(docs, ranked, docs_of, lang_of, n_docs=0, doc_quota=0.0):
    """Submitted documents with the ZH/EN quota applied, then extended to n_docs with pool documents."""
    docs = list(docs)
    pool_docs = list(dict.fromkeys(d for cid in ranked for d in docs_of(cid)))
    if doc_quota:
        listed = set(docs)
        extra = [d for d in pool_docs if d not in listed and lang_of.get(d, 'vi') != 'vi']
        need = math.ceil(doc_quota * len(docs)) - sum(lang_of.get(d, 'vi') != 'vi' for d in docs)
        vi_slots = [i for i, d in enumerate(docs) if lang_of.get(d, 'vi') == 'vi']
        slots = vi_slots[len(vi_slots) - min(max(need, 0), len(extra), len(vi_slots)):]
        for i, doc in zip(slots, extra):   # lowest VI slots, best ZH/EN document first
            docs[i] = doc
    if n_docs > len(docs):
        listed = set(docs)
        docs += [d for d in pool_docs if d not in listed][:n_docs - len(docs)]
    return docs


def best_chunks(ranked, docs_of):
    """{doc: its pool chunks, best first}. Documents filled from beyond the reranked pool have none."""
    best = {}
    for cid in ranked:
        for doc in docs_of(cid):
            best.setdefault(doc, []).append(cid)
    return best


def doc_text(best, rows, by_text, lead_chars=400, n_chunks=1):
    """Text a document reranker sees: the start of the document text (title / lead) + its best pool chunks.

    best: the document's pool chunks, best first; the n_chunks best are shown in text order.
    """
    shown = sorted(best[:n_chunks], key=lambda c: rows[c]['order'])
    parts = [rows[c]['text'] for c in shown]
    lead = rows[by_text[rows[shown[0]]['sha']][0]]
    if lead['order'] != rows[shown[0]]['order']:
        parts.insert(0, lead['text'][:lead_chars])
    return '\n...\n'.join(parts)


def fuse_docs(candidates, scores, weights, w_base=1.0, k=RRF_K, offset=0):
    """Candidates (submitted order) re-ordered by weighted RRF of their position and reranker scores.

    scores: {column: {doc: score}}; a document without a score in a column gets no term from it.
    offset: submitted rank of the first candidate (when the documents above it are kept fixed).
    """
    fused = {doc: w_base / (k + offset + rank) for rank, doc in enumerate(candidates)}
    for column, weight in weights.items():
        scored = sorted((d for d in candidates if d in scores[column]), key=lambda d: -scores[column][d])
        for rank, doc in enumerate(scored):
            fused[doc] += weight / (k + rank)
    return sorted(candidates, key=lambda d: -fused[d])


def spans(chosen, rows):
    """Merge chosen chunks of the same text with consecutive orders. Returns [(first chunk id, text)] in rank order."""
    by_sha = {}
    for cid in chosen:
        by_sha.setdefault(rows[cid]['sha'], []).append(cid)
    first_rank = {cid: i for i, cid in enumerate(chosen)}
    out = []
    for cids in by_sha.values():
        cids = sorted(cids, key=lambda c: rows[c]['order'])
        run = [cids[0]]
        for c in cids[1:]:
            if rows[c]['order'] == rows[run[-1]]['order'] + 1:
                run.append(c)
            else:
                out.append(run)
                run = [c]
        out.append(run)
    out.sort(key=lambda run: min(first_rank[c] for c in run))
    return [(run[0], merge_run([rows[c] for c in run])) for run in out]


def build_whole_docs(ranked, docs, budget, rows, by_text, docs_of):
    """Probe: every chunk of the submitted documents, best document first, until the budget is used.

    Tells whether chunk recall is lost inside the right documents (recall jumps) or by missing documents.
    """
    text_of = {}
    for cid in ranked:
        for doc in docs_of(cid):
            text_of.setdefault(doc, rows[cid]['sha'])
    chosen, used = [], 0
    for doc in docs:
        if doc not in text_of:
            continue
        for c in by_text[text_of[doc]]:
            cost = len(rows[c]['text'].encode('utf-8')) * len(docs_of(c))
            if used + cost > budget:
                return chosen
            if c not in chosen:
                chosen.append(c)
                used += cost
    return chosen


def load_inputs(pool_path=None, base=None, chunk_files=None):
    """Pool ranking, base submission, every chunk of the candidate texts and the document groups.

    The hybrid notebook passes its own pool.parquet, in-memory submission and validated chunk parts;
    otherwise they are found under INPUT.
    """
    pool = pd.read_parquet(pool_path or one_match('pool.parquet'),
                           columns=['query_id', 'chunk_id', 'final_rank', 'final_score'])
    pool = pool.sort_values(['query_id', 'final_rank'])
    ranked = {qid: g['chunk_id'].tolist() for qid, g in pool.groupby('query_id', sort=False)}
    score = {qid: g['final_score'].tolist() for qid, g in pool.groupby('query_id', sort=False)}
    base = load_submission() if base is None else base
    needed_sha_prefix = {cid.rsplit('-', 1)[0] for ids in ranked.values() for cid in ids}

    # Every chunk of the candidate texts: the neighbours are rows of the same text_sha256.
    rows, by_pos = {}, {}
    chunk_files = chunk_files or sorted(glob.glob(f'{INPUT}/**/chunks/*/part-*.parquet', recursive=True))
    assert chunk_files, 'no chunk parts: attach medxir-chunk-corpus'
    columns = ['chunk_id', 'text_sha256', 'rep_doc_id', 'order', 'start', 'end', 'chunk_text']
    for path in chunk_files:
        table = pq.read_table(path, columns=columns)
        prefix = pc.utf8_slice_codeunits(table['text_sha256'], 0, 16)
        table = table.filter(pc.is_in(prefix, value_set=pa.array(sorted(needed_sha_prefix))))
        lang = Path(path).parent.name   # chunks/<lang>/part-*.parquet
        for cid, sha, doc, order, start, end, text in zip(*(table[c].to_pylist() for c in columns)):
            rows[cid] = dict(sha=sha, doc=int(doc), order=int(order), start=int(start), end=int(end), text=text,
                             lang=lang)
            by_pos[(sha, int(order))] = cid
        print(Path(path).parent.name, Path(path).name, f'{len(rows):,} rows kept', flush=True)
    missing = {cid for ids in ranked.values() for cid in ids} - set(rows)
    assert not missing, f'{len(missing)} pool chunks not in the chunk corpus (chunk id != sha prefix?), e.g. {sorted(missing)[:3]}'

    groups = pd.read_parquet(one_match('hash_groups.parquet'), columns=['text_sha256', 'doc_ids', 'fanout_safe'])
    group = {sha: [int(d) for d in ids] for sha, ids, safe in
             zip(groups['text_sha256'], groups['doc_ids'], groups['fanout_safe']) if bool(safe)}
    del groups

    def docs_of(cid):
        return group.get(rows[cid]['sha']) or [rows[cid]['doc']]

    by_text = {}
    for cid in sorted(rows, key=lambda c: rows[c]['order']):
        by_text.setdefault(rows[cid]['sha'], []).append(cid)
    return dict(ranked=ranked, score=score, base=base, rows=rows, by_pos=by_pos, by_text=by_text, docs_of=docs_of,
                lang_of={doc: row['lang'] for cid, row in rows.items() for doc in docs_of(cid)})


def load_doc_scores():
    """{query_id: {column: {doc: score}}} from doc_rank.parquet (OUT_DIR first, then the inputs)."""
    path = OUT / 'doc_rank.parquet'
    frame = pd.read_parquet(path if path.exists() else one_match('doc_rank.parquet'))
    columns = [c for c in frame.columns if c not in ('query_id', 'doc_id', 'base_rank')]
    out = {}
    for qid, g in frame.groupby('query_id'):
        out[int(qid)] = {c: {int(d): float(s) for d, s in zip(g['doc_id'], g[c]) if s == s} for c in columns}
    return out


def confidence_budget(budget, score, gamma, threshold):
    """Per-query budgets re-weighted by (confidence / median) ** -gamma, the total kept."""
    conf = {qid: sum(s >= threshold for s in score.get(qid, [])) + 1 for qid in budget}
    median = sorted(conf.values())[len(conf) // 2]
    weight = {qid: min(max((c / median) ** -gamma, CONF_CLIP[0]), CONF_CLIP[1]) for qid, c in conf.items()}
    norm = sum(budget.values()) / sum(budget[q] * weight[q] for q in budget)
    return {qid: budget[qid] * weight[qid] * norm for qid in budget}


def cover_order(ranked, n_head, docs, first, last, docs_of):
    """Pool order with the best chunk of each listed document at ranks first..last moved right after the top n_head
    chunks, unless a top chunk already belongs to it."""
    head = ranked[:n_head]
    seen = {d for c in head for d in docs_of(c)}
    best = {}
    for cid in ranked[n_head:]:
        for doc in docs_of(cid):
            best.setdefault(doc, cid)
    promoted = list(dict.fromkeys(best[d] for d in docs[first:last] if d in best and d not in seen))
    moved = set(promoted)
    return head + promoted + [c for c in ranked[n_head:] if c not in moved]


def chosen_bytes(chosen, rows, docs_of):
    return sum(len(rows[c]['text'].encode('utf-8')) * len(docs_of(c)) for c in chosen)


def score_cut_budget(budget, ctx, n_neighbour, min_score):
    """Budgets after the score cut: what cut queries leave unused goes to the queries the cut did not stop,
    in proportion to their budgets."""
    used, full = {}, []
    for qid, b in budget.items():
        args = (ctx['ranked'][qid], n_neighbour, b, ctx['rows'], ctx['by_pos'], ctx['docs_of'])
        cut = chosen_bytes(build_query(*args, scores=ctx['score'][qid], min_score=min_score), ctx['rows'], ctx['docs_of'])
        if cut < chosen_bytes(build_query(*args), ctx['rows'], ctx['docs_of']):
            used[qid] = cut
        else:
            full.append(qid)
    freed = sum(budget[q] - used[q] for q in used)
    grow = 1 + freed / max(sum(budget[q] for q in full), 1)
    print(f'score cut {min_score}: {len(used)} queries cut, {freed / 1e6:.1f} MB moved to {len(full)} queries', flush=True)
    return {qid: used[qid] if qid in used else budget[qid] * grow for qid in budget}


def build_variants(ctx, only=None, doc_scores=None):
    ranked, base, rows, by_pos, by_text, docs_of, lang_of = (
        ctx[k] for k in ('ranked', 'base', 'rows', 'by_pos', 'by_text', 'docs_of', 'lang_of'))
    base_bytes = {e['id']: sum(len(c['chunk_text'].encode('utf-8')) for c in e['relevant_chunks'])
                  for e in base}
    only = ONLY if only is None else only
    manifest = {'budget_scale': BUDGET_SCALE, 'variants': {}}
    for name, spec in VARIANTS.items():
        if only and name not in only:
            continue
        if spec.get('doc_rr'):
            doc_scores = doc_scores or load_doc_scores()
            some = next(iter(doc_scores.values()))
            absent = [c for c in spec['doc_rr'] if c not in some]
            if absent:
                print(name, 'skipped: no doc_rank column', absent, flush=True)
                continue
        n_neighbour = spec['nb']
        budget = {qid: b * spec.get('budget_scale', BUDGET_SCALE) for qid, b in base_bytes.items()}
        q_budget = (confidence_budget(budget, ctx['score'], spec['conf_gamma'], spec.get('conf_t', 0.5))
                    if spec.get('conf_gamma') else budget)
        if spec.get('min_score'):
            q_budget = score_cut_budget(q_budget, ctx, n_neighbour, spec['min_score'])
        scale = 1.0
        while True:
            submission, n_spans, n_chunks, n_docs, other_docs, other_bytes, all_bytes, moved = [], 0, 0, 0, 0, 0, 0, 0
            dropped = 0
            for entry in base:
                qid = entry['id']
                if spec.get('doc_rr'):
                    q_scores = doc_scores.get(qid, {})
                    candidates = list(dict.fromkeys(
                        doc_list(entry['relevant_docs'], ranked[qid], docs_of, lang_of, n_docs=10 ** 6)
                        + [d for c in spec['doc_rr'] for d in q_scores.get(c, {})]))
                    keep = spec.get('n_docs') or len(entry['relevant_docs'])
                    fixed = candidates[:spec.get('keep_top', 0)]
                    docs = fixed + fuse_docs(candidates[len(fixed):], q_scores, spec['doc_rr'], spec.get('w_base', 1.0),
                                             offset=len(fixed))[:keep - len(fixed)]
                    moved += len(set(docs) - set(entry['relevant_docs']))
                else:
                    docs = doc_list(entry['relevant_docs'], ranked[qid], docs_of, lang_of,
                                    spec.get('n_docs', 0), spec.get('doc_quota', 0.0))
                if n_neighbour == 'whole':
                    chosen = build_whole_docs(ranked[qid], entry['relevant_docs'], q_budget[qid] * scale,
                                              rows, by_text, docs_of)
                elif spec.get('cover'):
                    order = cover_order(ranked[qid], n_neighbour, docs, *spec['cover'], docs_of)
                    chosen = build_query(order, n_neighbour, q_budget[qid] * scale, rows, by_pos, docs_of)
                else:
                    chosen = build_query(ranked[qid], n_neighbour, q_budget[qid] * scale, rows, by_pos, docs_of,
                                         spec.get('quota', 0.0), ctx['score'][qid], spec.get('min_score', 0.0))
                for c in chosen:
                    size = len(rows[c]['text'].encode('utf-8'))
                    all_bytes += size
                    other_bytes += size if rows[c]['lang'] != 'vi' else 0
                listed = set(docs)
                chunks = []
                for cid, text in spans(chosen, rows):
                    if len(text) < spec.get('min_chars', 0):
                        dropped += 1
                        continue
                    for doc in docs_of(cid):
                        chunks.append(dict(doc_id=doc, chunk_text=text, chunk_order=rows[cid]['order']))
                        if doc not in listed:
                            listed.add(doc)
                            docs.append(doc)
                n_spans += len(chunks)
                n_docs += len(docs)
                other_docs += sum(lang_of.get(d, 'vi') != 'vi' for d in docs) / max(len(docs), 1)
                n_chunks += len(chosen)
                submission.append(dict(id=qid, relevant_docs=docs, relevant_chunks=chunks))
            zip_path = OUT / f'medxir-expand-{name}-submission.zip'
            json_name = zip_path.name.replace('.zip', '.json')
            method = zipfile.ZIP_BZIP2 if spec.get('zip') == 'bzip2' else zipfile.ZIP_DEFLATED
            with zipfile.ZipFile(zip_path, 'w', compression=method, compresslevel=9) as archive:
                archive.writestr(json_name, json.dumps(submission, ensure_ascii=False, separators=(',', ':')))
            mb = zip_path.stat().st_size / 1e6
            if mb <= MAX_ZIP_MB:
                break
            scale *= 0.95 * MAX_ZIP_MB / mb
        stats = dict(spec, zip_mb=round(mb, 1), budget_scale_applied=round(scale, 3),
                     chunks_per_query=round(n_chunks / len(base), 1), entries_per_query=round(n_spans / len(base), 1),
                     docs_per_query=round(n_docs / len(base), 1), short_dropped_per_query=round(dropped / len(base), 1),
                     zh_en_doc_share=round(other_docs / len(base), 3),
                     zh_en_chunk_byte_share=round(other_bytes / max(all_bytes, 1), 3))
        if spec.get('doc_rr'):
            stats['new_docs_per_query'] = round(moved / len(base), 1)   # documents not in the submitted list
        manifest['variants'][name] = stats
        print(name, stats, flush=True)
    (OUT / 'expand_manifest.json').write_text(json.dumps(manifest, indent=2))
    return manifest


def main():
    build_variants(load_inputs())

## 1. Inputs, GPU check, pool and chunk rows

In [ ]:
if FAILED is None:
    with stage('setup'):
        import subprocess

        import numpy as np
        import torch

        for root in sorted(glob.glob('/kaggle/input/*/*/*')) or sorted(glob.glob('/kaggle/input/*')):
            print('input:', root, sorted(os.listdir(root))[:12] if os.path.isdir(root) else '')
        if DOC_RERANKERS:
            print(subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv'],
                                 capture_output=True, text=True).stdout)
            GPU_NAME = torch.cuda.get_device_name(0)
            assert '6000' in GPU_NAME, f'Expected RTX Pro 6000, got {GPU_NAME!r}'
        else:   # rebuild from an attached doc_rank.parquet: no scoring, CPU is enough
            assert PRIOR_DOC_RANK, 'DOC_RERANKERS is empty: set PRIOR_DOC_RANK = True and attach doc_rank.parquet'
            GPU_NAME = 'none'
        RERANK_PATH = {name: one_match(f'{name}/config.json').parent for name in DOC_RERANKERS}
        QUERY_TEXT = {int(q): str(t) for q, t in
                      pd.read_parquet(one_match('query.parquet'), columns=['id', 'query']).itertuples(index=False)} \
            if glob.glob(f'{INPUT}/**/query.parquet', recursive=True) else None

        CTX = load_inputs()
        if QUERY_TEXT is None:   # no query.parquet attached: the hybrid run's query_variants.json (qi = submission order)
            variants = json.loads(one_match('query_variants.json').read_text(encoding='utf-8'))['variants']
            QUERY_TEXT = {CTX['base'][v['qi']]['id']: v['text'] for v in variants if v['kind'] == 'full'}
        QIDS = [e['id'] for e in CTX['base']][:QUERY_LIMIT or None]
        assert all(q in QUERY_TEXT for q in QIDS), 'query text missing for some ids'
        PRIOR = None
        prior_hits = sorted(glob.glob(f'{INPUT}/**/doc_rank.parquet', recursive=True))
        assert DOC_RERANKERS or prior_hits, 'no doc_rank.parquet attached'
        if PRIOR_DOC_RANK and prior_hits:
            assert len(prior_hits) == 1, f'several doc_rank.parquet attached: {prior_hits}'
            PRIOR = pd.read_parquet(prior_hits[0])
            note(f'prior doc_rank.parquet: {len(PRIOR):,} rows, columns {list(PRIOR.columns)}')
        note(f'GPU {GPU_NAME}; {len(QIDS)} queries; {len(CTX["rows"]):,} chunk rows')

## 2. Candidate documents and their text

In [ ]:
if FAILED is None:
    with stage('candidates'):
        # Candidate documents per query and the text shown for each. Fan-out documents share a text: scored once.
        base_of = {e['id']: e for e in CTX['base']}
        n_max = max([end for _, end in DOC_RERANKERS.values()]
                    + ([int(PRIOR['base_rank'].max()) + 1] if PRIOR is not None else []))
        CANDS, TEXTS = {}, {}
        no_text = 0
        for qid in QIDS:
            ranked = CTX['ranked'][qid]
            best = best_chunks(ranked, CTX['docs_of'])
            cands = doc_list(base_of[qid]['relevant_docs'], ranked, CTX['docs_of'], CTX['lang_of'], n_docs=10 ** 6)[:n_max]
            CANDS[qid] = cands
            for doc in cands:
                if doc in best:
                    TEXTS[(qid, doc)] = doc_text(best[doc], CTX['rows'], CTX['by_text'], LEAD_CHARS, DOC_CHUNKS)
                else:
                    no_text += 1
        if PRIOR is not None:   # same pool + submission -> same candidate order; a mismatch means another input
            sample = PRIOR[PRIOR['query_id'].isin(QIDS[:50])]
            same = all(CANDS[q][r] == d for q, d, r in zip(sample['query_id'], sample['doc_id'], sample['base_rank'])
                       if r < len(CANDS[q]))
            assert same, 'prior doc_rank.parquet ranks other candidates: attach the matching hybrid run'
        note(f'{sum(map(len, CANDS.values())):,} candidate documents, {no_text:,} without a pool chunk (keep their rank only)')
        example = next(iter(TEXTS.values()))
        print('example document text:', example[:600].replace('\n', ' | '))

## 3. Rerank the documents (DOC_RERANKERS) -> doc_rank.parquet

In [ ]:
if FAILED is None:
    with stage('rerank'):
        from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer


        def load_scorer(name, path):
            if name.startswith('Qwen3-Reranker'):
                tok = AutoTokenizer.from_pretrained(str(path), padding_side='left')
                model = AutoModelForCausalLM.from_pretrained(str(path), torch_dtype=torch.bfloat16).cuda().eval()
                yes, no = tok.convert_tokens_to_ids('yes'), tok.convert_tokens_to_ids('no')
                prefix = ('<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the '
                          'Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n')
                suffix = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
                instruct = 'Given a Vietnamese medical question, judge whether the document answers the question'

                def score(queries, docs):
                    texts = [f'{prefix}<Instruct>: {instruct}\n<Query>: {q}\n<Document>: {d}{suffix}' for q, d in zip(queries, docs)]
                    enc = tok(texts, padding=True, truncation=True, max_length=DOC_MAX_TOKENS, return_tensors='pt').to('cuda')
                    logits = model(**enc).logits[:, -1, :]
                    return torch.softmax(torch.stack([logits[:, no], logits[:, yes]], dim=1).float(), dim=1)[:, 1]
                return model, score, QWEN_BATCH
            tok = AutoTokenizer.from_pretrained(str(path))
            model = AutoModelForSequenceClassification.from_pretrained(str(path), torch_dtype=torch.bfloat16).cuda().eval()

            def score(queries, docs):
                enc = tok(queries, docs, padding=True, truncation=True, max_length=DOC_MAX_TOKENS, return_tensors='pt').to('cuda')
                return torch.sigmoid(model(**enc).logits.view(-1).float())
            return model, score, 64


        SCORES = {}   # reranker -> {(qid, doc): score}
        for name, (first, last) in DOC_RERANKERS.items():
            started = time.perf_counter()
            model, score, batch = load_scorer(name, RERANK_PATH[name])
            keys = {}   # (qid, text) -> documents sharing it
            for qid in QIDS:
                for doc in CANDS[qid][first:last]:
                    if (qid, doc) in TEXTS:
                        keys.setdefault((qid, TEXTS[(qid, doc)]), []).append(doc)
            pairs = list(keys)
            order = np.argsort([len(t) for _, t in pairs], kind='stable')   # similar lengths per batch: less padding
            values = np.empty(len(pairs), dtype=np.float32)
            note(f'{name}: {len(pairs):,} pairs')
            t0, next_report = time.perf_counter(), 0.05
            with torch.inference_mode():
                for start in range(0, len(order), batch):
                    index = order[start:start + batch]
                    values[index] = score([QUERY_TEXT[pairs[i][0]] for i in index], [pairs[i][1] for i in index]).cpu().numpy()
                    done = (start + len(index)) / len(order)
                    if done >= next_report:
                        rate = (start + len(index)) / (time.perf_counter() - t0)
                        note(f'{name} {done:.0%} | {rate:.0f} pairs/s | ETA {(len(order) - start) / rate / 60:.0f} min')
                        next_report += 0.05
            SCORES[name] = {(qid, doc): float(v) for (qid, _), v in zip(pairs, values) for doc in keys[(qid, _)]}
            del model
            torch.cuda.empty_cache()
            note(f'{name} done in {time.perf_counter() - started:.0f}s')

        rows_out = []
        for qid in QIDS:
            for rank, doc in enumerate(CANDS[qid]):
                rows_out.append(dict(query_id=qid, doc_id=doc, base_rank=rank,
                                     **{name: SCORES[name].get((qid, doc), np.nan) for name in DOC_RERANKERS}))
        frame = pd.DataFrame(rows_out)
        if PRIOR is not None:
            keep = [c for c in PRIOR.columns if c not in frame.columns or c in ('query_id', 'doc_id')]
            frame = frame.merge(PRIOR[keep], on=['query_id', 'doc_id'], how='outer')
            frame['base_rank'] = frame['base_rank'].fillna(-1).astype(int)
        frame.to_parquet(OUT / 'doc_rank.parquet', index=False, compression='zstd')
        note(f'doc_rank.parquet: {len(frame):,} rows, columns {list(frame.columns)}')

## 4. nb60 submissions with fused document lists

In [ ]:
if FAILED is None:
    with stage('build'):
        if QUERY_LIMIT:
            print('QUERY_LIMIT set: no submission written')
        else:
            manifest = build_variants(CTX, only=BUILD, doc_scores=load_doc_scores())
            manifest.update(doc_rerankers=DOC_RERANKERS, lead_chars=LEAD_CHARS, doc_max_tokens=DOC_MAX_TOKENS, gpu=GPU_NAME,
                            elapsed_s=round(time.perf_counter() - T0, 1))
            (OUT / 'expand_manifest.json').write_text(json.dumps(manifest, indent=2))
            print(json.dumps(manifest, indent=2))

## 5. Report

In [ ]:
note('finished; status ' + ('error' if FAILED else 'ok'))
if FAILED:
    (OUT / 'stage_status.json').write_text(json.dumps(
        dict(status='error', failed_stage=FAILED, error_file='retrieval_error.txt'), indent=2) + '\n')